call h5ad

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import anndata as ad
from scipy import sparse
from sklearn.preprocessing import OneHotEncoder

In [2]:
import matplotlib.pyplot as plt

def plot_histogram(arr):
    plt.hist(arr, bins=30)
    plt.xlabel("Value")
    plt.ylabel("Count")
    plt.title("Histogram")
    plt.show()

In [3]:
folder = "/home/kchen/microbiome/gut_microbiome_GPT/datasets/metagenomics"

cmw = ad.read_h5ad(os.path.join(folder, "merged_curatedMD.h5ad"))
gmwi = ad.read_h5ad(os.path.join(folder, "gmwi/gmwi_all.h5ad"))

/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [4]:
print(cmw)
print(gmwi)

AnnData object with n_obs × n_vars = 22584 × 2055
    obs: 'study_name'
AnnData object with n_obs × n_vars = 9798 × 3434
    obs: 'Sample Accession', 'Study_ID'


In [5]:
display(cmw.obs["study_name"])
display(gmwi.obs["Study_ID"])

AsnicarF_2021_ERR4330026                                                                                         AsnicarF_2021
AsnicarF_2021_ERR4330027                                                                                         AsnicarF_2021
AsnicarF_2021_ERR4330028                                                                                         AsnicarF_2021
AsnicarF_2021_ERR4330029                                                                                         AsnicarF_2021
AsnicarF_2021_ERR4330030                                                                                         AsnicarF_2021
                                                                                                                   ...        
MetaCardis_2020_a_ERR4091525;ERR4091526;ERR4091527;ERR4091528;ERR4091529;ERR4091530;ERR4091531               MetaCardis_2020_a
MetaCardis_2020_a_ERR4565737;ERR4565738;ERR4565739;ERR4565740;ERR4565741;ERR4565742;ERR4565743;ERR4565744    Me

study_id
Palleja (2018)    Palleja (2018)
Palleja (2018)    Palleja (2018)
Palleja (2018)    Palleja (2018)
Palleja (2018)    Palleja (2018)
Palleja (2018)    Palleja (2018)
                       ...      
Yang (2020)          Yang (2020)
Yang (2020)          Yang (2020)
Yang (2020)          Yang (2020)
Yang (2020)          Yang (2020)
Yang (2020)          Yang (2020)
Name: Study_ID, Length: 9798, dtype: category
Categories (64, object): ['Ananthakrishnan (2017)', 'Ang (2021)', 'Asnicar (2021)', 'Backhed (2015)', ..., 'Zhang (2015)', 'Zhu (2021)', 'Zysset-Burri (2019)', 'de Nies (2023)']

In [6]:
import re

def parse_study_label(s: str) -> str:
    """
    Convert 'AsnicarF (2021)' -> 'AsnicarF_2021'
    """
    m = re.fullmatch(r"(.+?)\s*\((\d{4})\)", str(s))
    if not m:
        return s  # fallback: unchanged
    name, year = m.groups()
    return f"{name}_{year}"


gmwi.obs["study_name"] = gmwi.obs["Study_ID"].apply(parse_study_label)
gmwi.obs_names = gmwi.obs["Study_ID"].apply(parse_study_label)
gmwi.obs_names = (
    gmwi.obs_names.astype(str)
    + "_"
    + gmwi.obs["Sample Accession"].astype(str)
)
gmwi.obs = gmwi.obs.drop(columns=["Study_ID"])
gmwi.obs = gmwi.obs.drop(columns=["Sample Accession"])



/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:812: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    ['Palleja_2018', 'Palleja_2018', 'Palleja_2018', 'Palleja_2018', 'Palleja_2018']

    Inferred to be: categorical

  names = self._prep_dim_index(names, "obs")


In [7]:
print(cmw)
print(gmwi)
display(gmwi.obs["study_name"])

AnnData object with n_obs × n_vars = 22584 × 2055
    obs: 'study_name'
AnnData object with n_obs × n_vars = 9798 × 3434
    obs: 'study_name'


Palleja_2018_SAMEA104062441    Palleja_2018
Palleja_2018_SAMEA104062442    Palleja_2018
Palleja_2018_SAMEA104062443    Palleja_2018
Palleja_2018_SAMEA104062444    Palleja_2018
Palleja_2018_SAMEA104062445    Palleja_2018
                                   ...     
Yang_2020_SRR6456373              Yang_2020
Yang_2020_SRR6456374              Yang_2020
Yang_2020_SRR6456375              Yang_2020
Yang_2020_SRR6456376              Yang_2020
Yang_2020_SRR6456377              Yang_2020
Name: study_name, Length: 9798, dtype: category
Categories (64, object): ['Ananthakrishnan_2017', 'Ang_2021', 'Asnicar_2021', 'Backhed_2015', ..., 'Zhang_2015', 'Zhu_2021', 'Zysset-Burri_2019', 'de Nies_2023']

check for duplicate study names

In [8]:
cmw_studies = set(cmw.obs["study_name"])
gmwi_studies = set(gmwi.obs["study_name"])

shared_studies = cmw_studies & gmwi_studies
print(f"Shared studies: {shared_studies}")


Shared studies: set()


check match between cmw and gmwi taxa

In [9]:
cmw_cols = set(cmw.var_names)
gmwi_cols = set(gmwi.var_names)

shared_cols = cmw_cols & gmwi_cols
cmw_only = cmw_cols - shared_cols
gmwi_only = gmwi_cols - shared_cols

print("Shared columns:", len(shared_cols))
print("CMW only columns:", len(cmw_only))
print("GMWI only columns:", len(gmwi_only))

Shared columns: 1440
CMW only columns: 615
GMWI only columns: 1994


In [10]:
joined = ad.concat([cmw, gmwi], axis=0, join="outer")

/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [11]:
print(joined.var_names)

Index(['UNKNOWN.1', 'k__Archaea', 'k__Archaea|p__Euryarchaeota',
       'k__Archaea|p__Euryarchaeota|c__Methanobacteria',
       'k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales',
       'k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales|f__Methanobacteriaceae',
       'k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales|f__Methanobacteriaceae|g__Methanobacterium',
       'k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales|f__Methanobacteriaceae|g__Methanobacterium|s__Methanobacterium_formicicum',
       'k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales|f__Methanobacteriaceae|g__Methanobacterium|s__Methanobacterium_sp_MB1',
       'k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales|f__Methanobacteriaceae|g__Methanobrevibacter',
       ...
       'k__Viruses|p__Viruses_unclassified|c__Viruses_unclassified|o__Viruses_unclassified|f__Viruses_unclassified|g__Viruses_unclassified

In [12]:
joined.X = joined.X.tocsr()
row_sums = np.asarray(joined.X.sum(axis=1)).ravel()
row_sums[row_sums == 0] = 1.0
joined.X = joined.X.multiply(100.0 / row_sums[:, None])



/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/storage.py:48: FutureWarning: AnnData previously had undefined behavior around matrices of type <class 'scipy.sparse._coo.coo_matrix'>.In 0.12, passing in this type will throw an error. Please convert to a supported type.Continue using for this minor version at your own risk.
  warnings.warn(msg, FutureWarning)


In [13]:
joined

AnnData object with n_obs × n_vars = 32382 × 4049
    obs: 'study_name'

add Taxonomy split

In [14]:
def qiime_is_prefix(short_tax: str, long_tax: str) -> bool:
    """
    True if short_tax is a rank-prefix of long_tax (QIIME taxonomy strings).
    """
    short_parts = [p.strip() for p in short_tax.split("|") if p.strip()]
    long_parts  = [p.strip() for p in long_tax.split("|") if p.strip()]
    return len(short_parts) <= len(long_parts) and long_parts[:len(short_parts)] == short_parts

a = "k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales"
b = "k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales|f__Methanobacteriaceae|g__Methanosphaera|s__Methanosphaera_cuniculi"

def split_dict(d, predicate):
    yes, no = {}, {}
    for k, v in d.items():
        (yes if predicate(k, v[0]) else no)[k] = v
    return yes, no




In [15]:
import pandas as pd

RANK_MAP = {
    "k": "Domain",
    "p": "Phylum",
    "c": "Class",
    "o": "Order",
    "f": "Family",
    "g": "Genus",
    "s": "Species",
}

def parse_qiime_taxonomy(tax: str) -> dict:
    """
    Parse a QIIME taxonomy string into rank columns.
    Missing ranks are filled with None.
    """
    out = {v: None for v in RANK_MAP.values()}

    for part in tax.split("|"):
        part = part.strip()
        if "__" not in part:
            continue
        rank, value = part.split("__", 1)
        if rank in RANK_MAP and value:
            out[RANK_MAP[rank]] = value

    return out

parse_qiime_taxonomy("k__Bacteria|p__Firmicutes|c__Bacilli|o__Lactobacillales|f__Lactobacillaceae|g__Lactobacillus")

{'Domain': 'Bacteria',
 'Phylum': 'Firmicutes',
 'Class': 'Bacilli',
 'Order': 'Lactobacillales',
 'Family': 'Lactobacillaceae',
 'Genus': 'Lactobacillus',
 'Species': None}

In [16]:
def qiime_taxonomy_table(qiime_names):
    rows = [parse_qiime_taxonomy(t) for t in qiime_names]
    df = pd.DataFrame(rows, index=qiime_names)
    df.index.name = "qiime_taxonomy"
    return df


joined.varm["taxonomy"] = qiime_taxonomy_table(joined.var_names)

In [17]:
joined.varm["taxonomy"]

,Domain,Phylum,Class,Order,Family,Genus,Species
qiime_taxonomy,,,,,,,
UNKNOWN.1,None,None,None,None,None,None,None
k__Archaea,Archaea,None,None,None,None,None,None
k__Archaea|p__Euryarchaeota,Archaea,Euryarchaeota,None,None,None,None,None
k__Archaea|p__Euryarchaeota|c__Methanobacteria,Archaea,Euryarchaeota,Methanobacteria,None,None,None,None
k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales,Archaea,Euryarchaeota,Methanobacteria,Methanobacteriales,None,None,None
...,...,...,...,...,...,...,...
k__Viruses|p__Viruses_unclassified|c__Viruses_unclassified|o__Viruses_unclassified|f__Viruses_unclassified|g__Viruses_unclassified|s__Staphylococcus_phage_PT1028,Viruses,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Staphylococcus_phage_PT1028
k__Viruses|p__Viruses_unclassified|c__Viruses_unclassified|o__Viruses_unclassified|f__Viruses_unclassified|g__Viruses_unclassified|s__Streptococcus_pyogenes_phage_5005_1,Viruses,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Streptococcus_pyogenes_phage_5005_1
k__Viruses|p__Viruses_unclassified|c__Viruses_unclassified|o__Viruses_unclassified|f__Viruses_unclassified|g__Viruses_unclassified|s__Streptococcus_pyogenes_phage_5005_2,Viruses,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Streptococcus_pyogenes_phage_5005_2


Check for duplicate studies

In [18]:
joined

AnnData object with n_obs × n_vars = 32382 × 4049
    obs: 'study_name'
    varm: 'taxonomy'

In [19]:
joined.obs["study_name"]

AsnicarF_2021_ERR4330026    AsnicarF_2021
AsnicarF_2021_ERR4330027    AsnicarF_2021
AsnicarF_2021_ERR4330028    AsnicarF_2021
AsnicarF_2021_ERR4330029    AsnicarF_2021
AsnicarF_2021_ERR4330030    AsnicarF_2021
                                ...      
Yang_2020_SRR6456373            Yang_2020
Yang_2020_SRR6456374            Yang_2020
Yang_2020_SRR6456375            Yang_2020
Yang_2020_SRR6456376            Yang_2020
Yang_2020_SRR6456377            Yang_2020
Name: study_name, Length: 32382, dtype: category
Categories (157, object): ['Ananthakrishnan_2017', 'Ang_2021', 'AsnicarF_2017', 'AsnicarF_2021', ..., 'ZhuF_2020', 'Zhu_2021', 'Zysset-Burri_2019', 'de Nies_2023']

looking at match between 16S and shotgun data taxonomy

In [20]:
test_finetune = ad.read_h5ad("/home/kchen/microbiome/gut_microbiome_GPT/datasets/remove_redundant_oct24/finetune_test.h5ad")
old_taxonomy = test_finetune.varm["taxonomy"]
display(old_taxonomy)

/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


,Domain,Phylum,Class,Order,Family,Genus
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Atopobiaceae.Tractidigestivibacter,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Atopobiaceae,Tractidigestivibacter
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Coriobacteriaceae.Collinsella,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Coriobacteriaceae,Collinsella
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Adlercreutzia,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Eggerthellaceae,Adlercreutzia
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Senegalimassilia,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Eggerthellaceae,Senegalimassilia
Bacteria.Bacillota.Bacilli.Erysipelotrichales.Erysipelotrichaceae.Holdemanella,Bacteria,Bacillota,Bacilli,Erysipelotrichales,Erysipelotrichaceae,Holdemanella
...,...,...,...,...,...,...
Bacteria.Bacteroidota.Bacteroidia.Flavobacteriales.Flavobacteriaceae.Flavirhabdus,Bacteria,Bacteroidota,Bacteroidia,Flavobacteriales,Flavobacteriaceae,Flavirhabdus
Bacteria.Pseudomonadota.Alphaproteobacteria.Rhodobacterales.Paracoccaceae.Octadecabacter,Bacteria,Pseudomonadota,Alphaproteobacteria,Rhodobacterales,Paracoccaceae,Octadecabacter
Bacteria.Pseudomonadota.Alphaproteobacteria.Acetobacterales.Acetobacteraceae.Swingsia,Bacteria,Pseudomonadota,Alphaproteobacteria,Acetobacterales,Acetobacteraceae,Swingsia
Bacteria.Bacteroidota.Bacteroidia.Flavobacteriales.Flavobacteriaceae.Aurantiacicella,Bacteria,Bacteroidota,Bacteroidia,Flavobacteriales,Flavobacteriaceae,Aurantiacicella


In [21]:
display(joined.varm["taxonomy"])

,Domain,Phylum,Class,Order,Family,Genus,Species
qiime_taxonomy,,,,,,,
UNKNOWN.1,None,None,None,None,None,None,None
k__Archaea,Archaea,None,None,None,None,None,None
k__Archaea|p__Euryarchaeota,Archaea,Euryarchaeota,None,None,None,None,None
k__Archaea|p__Euryarchaeota|c__Methanobacteria,Archaea,Euryarchaeota,Methanobacteria,None,None,None,None
k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales,Archaea,Euryarchaeota,Methanobacteria,Methanobacteriales,None,None,None
...,...,...,...,...,...,...,...
k__Viruses|p__Viruses_unclassified|c__Viruses_unclassified|o__Viruses_unclassified|f__Viruses_unclassified|g__Viruses_unclassified|s__Staphylococcus_phage_PT1028,Viruses,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Staphylococcus_phage_PT1028
k__Viruses|p__Viruses_unclassified|c__Viruses_unclassified|o__Viruses_unclassified|f__Viruses_unclassified|g__Viruses_unclassified|s__Streptococcus_pyogenes_phage_5005_1,Viruses,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Streptococcus_pyogenes_phage_5005_1
k__Viruses|p__Viruses_unclassified|c__Viruses_unclassified|o__Viruses_unclassified|f__Viruses_unclassified|g__Viruses_unclassified|s__Streptococcus_pyogenes_phage_5005_2,Viruses,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Streptococcus_pyogenes_phage_5005_2


In [22]:
joined.varm["taxonomy"]["Phylum"] = joined.varm["taxonomy"]["Phylum"].replace("Actinobacteria", "Actinomycetota").replace("Proteobacteria", "Pseudomonadota").replace("Firmicutes", "Bacillota")

In [23]:
def check_row(row: pd.Series) -> bool:
  
    parts = [row[k] for k in ["Domain", "Phylum", "Class", "Order", "Family", "Genus", "Species"]]
    
    non_null_count = sum(1 for p in parts[:-1] if p is not None)
    match_count = 0
    last_match_index = -1
    match_row = -1
    for i in range(min(non_null_count, old_taxonomy.shape[1])):
        if parts[i] in old_taxonomy.iloc[:, i].to_list():
            match_count += 1
            last_match_index = i
            match_row = old_taxonomy.loc[old_taxonomy.iloc[:, i] == parts[i]]
    return last_match_index, match_count, non_null_count, match_row

def run_match():
    last_match_index_list, match_count_list, non_null_count_list, match_rows = [], [], [], []
    for idx, row in joined.varm["taxonomy"].iterrows():
        last_match_index, matched_ranks, total_ranks, match_row = check_row(row)
        last_match_index_list.append(last_match_index)
        match_count_list.append(matched_ranks)
        non_null_count_list.append(total_ranks)
        match_rows.append(match_row)

    last_match_indexes = pd.Series(last_match_index_list)
    match_counts = pd.Series(match_count_list)
    non_null_counts = pd.Series(non_null_count_list)
    match_rows = pd.Series(match_rows)
    return last_match_indexes, match_counts, non_null_counts, match_rows

# check_row(joined.varm["taxonomy"].iloc[2])

In [24]:
last_match_indexes, match_counts, non_null_counts, match_rows = run_match()

In [25]:
final_match = (non_null_counts - last_match_indexes - 1 == 0)
pos_idx = np.flatnonzero(final_match.to_numpy())

# final_match.reindex(joined.varm["taxonomy"].index)

In [26]:
(non_null_counts - last_match_indexes - 1).value_counts()

0    2641
6     953
1     194
5     121
2      61
4      40
3      39
Name: count, dtype: int64

In [27]:
len(final_match)

4049

In [28]:
len(pos_idx)

2641

In [29]:
(match_counts == non_null_counts).sum()

1402

In [30]:
joined.varm["taxonomy"].iloc[pos_idx]

,Domain,Phylum,Class,Order,Family,Genus,Species
qiime_taxonomy,,,,,,,
UNKNOWN.1,None,None,None,None,None,None,None
k__Archaea,Archaea,None,None,None,None,None,None
k__Archaea|p__Euryarchaeota|c__Methanobacteria,Archaea,Euryarchaeota,Methanobacteria,None,None,None,None
k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales,Archaea,Euryarchaeota,Methanobacteria,Methanobacteriales,None,None,None
k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales|f__Methanobacteriaceae,Archaea,Euryarchaeota,Methanobacteria,Methanobacteriales,Methanobacteriaceae,None,None
...,...,...,...,...,...,...,...
k__Bacteria|p__Verrucomicrobia|c__Verrucomicrobiae|o__Verrucomicrobiales|f__Akkermansiaceae,Bacteria,Verrucomicrobia,Verrucomicrobiae,Verrucomicrobiales,Akkermansiaceae,None,None
k__Bacteria|p__Verrucomicrobia|c__Verrucomicrobiae|o__Verrucomicrobiales|f__Akkermansiaceae|g__Akkermansia,Bacteria,Verrucomicrobia,Verrucomicrobiae,Verrucomicrobiales,Akkermansiaceae,Akkermansia,None
k__Bacteria|p__Verrucomicrobia|c__Verrucomicrobiae|o__Verrucomicrobiales|f__Akkermansiaceae|g__Akkermansia|s__Akkermansia_glycaniphila,Bacteria,Verrucomicrobia,Verrucomicrobiae,Verrucomicrobiales,Akkermansiaceae,Akkermansia,Akkermansia_glycaniphila


Doing the merge. rule: if they have completely the same taxonomy table, they should be the same taxa

In [31]:
bool_idx = ~(match_counts == non_null_counts) & (non_null_counts - last_match_indexes - 1)
pos_idx = np.flatnonzero(bool_idx.to_numpy())

joined.varm["taxonomy"].iloc[pos_idx]

,Domain,Phylum,Class,Order,Family,Genus,Species
qiime_taxonomy,,,,,,,
k__Archaea|p__Euryarchaeota,Archaea,Euryarchaeota,None,None,None,None,None
k__Archaea|p__Euryarchaeota|c__Thermococci|o__Thermococcales|f__Thermococcaceae|g__Pyrococcus,Archaea,Euryarchaeota,Thermococci,Thermococcales,Thermococcaceae,Pyrococcus,None
k__Archaea|p__Euryarchaeota|c__Thermococci|o__Thermococcales|f__Thermococcaceae|g__Pyrococcus|s__Pyrococcus_yayanosii,Archaea,Euryarchaeota,Thermococci,Thermococcales,Thermococcaceae,Pyrococcus,Pyrococcus_yayanosii
k__Archaea|p__Euryarchaeota|c__Thermoplasmata|o__Methanomassiliicoccales|f__Methanomassiliicoccaceae|g__Candidatus_Methanomethylophilus,Archaea,Euryarchaeota,Thermoplasmata,Methanomassiliicoccales,Methanomassiliicoccaceae,Candidatus_Methanomethylophilus,None
k__Archaea|p__Euryarchaeota|c__Thermoplasmata|o__Methanomassiliicoccales|f__Methanomassiliicoccaceae|g__Candidatus_Methanomethylophilus|s__Candidatus_Methanomethylophilus_sp_1R26,Archaea,Euryarchaeota,Thermoplasmata,Methanomassiliicoccales,Methanomassiliicoccaceae,Candidatus_Methanomethylophilus,Candidatus_Methanomethylophilus_sp_1R26
...,...,...,...,...,...,...,...
k__Viruses|p__Viruses_unclassified|c__Viruses_unclassified|o__Viruses_unclassified|f__Togaviridae,Viruses,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Togaviridae,None,None
k__Viruses|p__Viruses_unclassified|c__Viruses_unclassified|o__Viruses_unclassified|f__Tombusviridae,Viruses,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Tombusviridae,None,None
k__Viruses|p__Viruses_unclassified|c__Viruses_unclassified|o__Viruses_unclassified|f__Totiviridae,Viruses,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Totiviridae,None,None


In [32]:
import scipy.sparse as sp

if sp.issparse(joined.X):
    joined.X = joined.X.tocsr()   # or .tocsc()

tax = joined.varm["taxonomy"].copy()

# bytes -> str
tax = tax.applymap(lambda x: x.decode("utf-8") if isinstance(x, (bytes, bytearray)) else x)

# fill missing, then force plain python strings (object dtype)
tax = tax.astype(object).applymap(str)

joined.varm["taxonomy"] = tax
joined.varm["taxonomy"]

/tmp/ipykernel_153980/155027931.py:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  tax = tax.applymap(lambda x: x.decode("utf-8") if isinstance(x, (bytes, bytearray)) else x)
/tmp/ipykernel_153980/155027931.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  tax = tax.astype(object).applymap(str)


,Domain,Phylum,Class,Order,Family,Genus,Species
qiime_taxonomy,,,,,,,
UNKNOWN.1,None,None,None,None,None,None,None
k__Archaea,Archaea,None,None,None,None,None,None
k__Archaea|p__Euryarchaeota,Archaea,Euryarchaeota,None,None,None,None,None
k__Archaea|p__Euryarchaeota|c__Methanobacteria,Archaea,Euryarchaeota,Methanobacteria,None,None,None,None
k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales,Archaea,Euryarchaeota,Methanobacteria,Methanobacteriales,None,None,None
...,...,...,...,...,...,...,...
k__Viruses|p__Viruses_unclassified|c__Viruses_unclassified|o__Viruses_unclassified|f__Viruses_unclassified|g__Viruses_unclassified|s__Staphylococcus_phage_PT1028,Viruses,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Staphylococcus_phage_PT1028
k__Viruses|p__Viruses_unclassified|c__Viruses_unclassified|o__Viruses_unclassified|f__Viruses_unclassified|g__Viruses_unclassified|s__Streptococcus_pyogenes_phage_5005_1,Viruses,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Streptococcus_pyogenes_phage_5005_1
k__Viruses|p__Viruses_unclassified|c__Viruses_unclassified|o__Viruses_unclassified|f__Viruses_unclassified|g__Viruses_unclassified|s__Streptococcus_pyogenes_phage_5005_2,Viruses,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Viruses_unclassified,Streptococcus_pyogenes_phage_5005_2


In [33]:
# joined.write_h5ad("/home/kchen/microbiome/gut_microbiome_GPT/datasets/metagenomics/merged_wgs_with_taxonomy.h5ad")

In [34]:
joined

AnnData object with n_obs × n_vars = 32382 × 4049
    obs: 'study_name'
    varm: 'taxonomy'

In [35]:
joined.X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3734693 stored elements and shape (32382, 4049)>